In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
# =========================================================
# 1. project_data 폴더 자동 탐색
# =========================================================

CURRENT_DIR = Path.cwd().resolve()

candidate_dirs = [
    CURRENT_DIR / "project_data",
    CURRENT_DIR.parent / "project_data",
    CURRENT_DIR.parent.parent / "project_data",
]

DATA_DIR = next(
    (path for path in candidate_dirs if path.exists()),
    None
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "project_data 폴더를 찾지 못했습니다.\n"
        f"현재 실행 위치: {CURRENT_DIR}"
    )

PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = (
    PROCESSED_DIR
    / "all_age_commute_morning_0700_0940.csv"
)

OUTPUT_FILE = (
    PROCESSED_DIR
    / "all_age_commute_od_aggregated.csv"
)

print("입력 파일:", INPUT_FILE)
print("입력 파일 존재:", INPUT_FILE.exists())
print("출력 파일:", OUTPUT_FILE)

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"입력 파일을 찾지 못했습니다: {INPUT_FILE}"
    )

입력 파일: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/all_age_commute_morning_0700_0940.csv
입력 파일 존재: True
출력 파일: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/all_age_commute_od_aggregated.csv


In [3]:
# =========================================================
# 2. 컬럼 설정
# =========================================================

USE_COLS = [
    "출발 행정동 코드",
    "출발 행정동",
    "도착 행정동 코드",
    "도착 행정동",
    "이동거리",
    "이동 시간",
    "이동 인구수",
]

DTYPES = {
    "출발 행정동 코드": "string",
    "출발 행정동": "string",
    "도착 행정동 코드": "string",
    "도착 행정동": "string",
    "이동거리": "float64",
    "이동 시간": "float64",
    "이동 인구수": "float64",
}

GROUP_COLS = [
    "출발 행정동 코드",
    "출발 행정동",
    "도착 행정동 코드",
    "도착 행정동",
]

In [4]:
# =========================================================
# 3. 청크 단위 OD 집계
# =========================================================

CHUNK_SIZE = 1_000_000

chunk_results = []

total_rows = 0
chunk_number = 0


reader = pd.read_csv(
    INPUT_FILE,
    usecols=USE_COLS,
    dtype=DTYPES,
    encoding="utf-8-sig",
    chunksize=CHUNK_SIZE,
    low_memory=False,
)


for chunk in reader:
    chunk_number += 1
    total_rows += len(chunk)

    # -----------------------------------------------------
    # 코드와 이름 공백 정리
    # -----------------------------------------------------

    for col in [
        "출발 행정동 코드",
        "출발 행정동",
        "도착 행정동 코드",
        "도착 행정동",
    ]:
        chunk[col] = chunk[col].str.strip()

    # -----------------------------------------------------
    # 집계에 필요한 핵심 값 결측 제거
    # -----------------------------------------------------

    chunk = chunk.dropna(
        subset=[
            "출발 행정동 코드",
            "도착 행정동 코드",
            "이동 인구수",
            "이동 시간",
            "이동거리",
        ]
    )

    # 이동량이 0 이하인 행은 가중평균 계산에 사용할 수 없으므로 제외
    chunk = chunk.loc[
        chunk["이동 인구수"] > 0
    ].copy()

    # -----------------------------------------------------
    # 가중합 계산
    # -----------------------------------------------------

    chunk["이동시간_가중합"] = (
        chunk["이동 시간"]
        * chunk["이동 인구수"]
    )

    chunk["이동거리_가중합"] = (
        chunk["이동거리"]
        * chunk["이동 인구수"]
    )

    # -----------------------------------------------------
    # 현재 청크 내부 OD 집계
    # -----------------------------------------------------

    chunk_od = (
        chunk
        .groupby(
            GROUP_COLS,
            as_index=False,
            dropna=False,
            observed=True,
        )
        .agg(
            출근_이동량=("이동 인구수", "sum"),
            이동시간_가중합=("이동시간_가중합", "sum"),
            이동거리_가중합=("이동거리_가중합", "sum"),
            원본_행수=("이동 인구수", "size"),
        )
    )

    chunk_results.append(chunk_od)

    print(
        f"{chunk_number:>3}번째 청크 완료 | "
        f"누적 원본 행 수: {total_rows:,} | "
        f"현재 청크 OD 수: {len(chunk_od):,}"
    )


print("\n1차 청크 집계 완료")
print("처리 원본 행 수:", f"{total_rows:,}")
print("생성된 청크 결과 수:", len(chunk_results))

  1번째 청크 완료 | 누적 원본 행 수: 1,000,000 | 현재 청크 OD 수: 103,443
  2번째 청크 완료 | 누적 원본 행 수: 2,000,000 | 현재 청크 OD 수: 103,639
  3번째 청크 완료 | 누적 원본 행 수: 3,000,000 | 현재 청크 OD 수: 103,535
  4번째 청크 완료 | 누적 원본 행 수: 4,000,000 | 현재 청크 OD 수: 103,828
  5번째 청크 완료 | 누적 원본 행 수: 5,000,000 | 현재 청크 OD 수: 104,262
  6번째 청크 완료 | 누적 원본 행 수: 6,000,000 | 현재 청크 OD 수: 103,796
  7번째 청크 완료 | 누적 원본 행 수: 7,000,000 | 현재 청크 OD 수: 104,030
  8번째 청크 완료 | 누적 원본 행 수: 8,000,000 | 현재 청크 OD 수: 104,146
  9번째 청크 완료 | 누적 원본 행 수: 9,000,000 | 현재 청크 OD 수: 103,987
 10번째 청크 완료 | 누적 원본 행 수: 10,000,000 | 현재 청크 OD 수: 104,319
 11번째 청크 완료 | 누적 원본 행 수: 11,000,000 | 현재 청크 OD 수: 104,062
 12번째 청크 완료 | 누적 원본 행 수: 12,000,000 | 현재 청크 OD 수: 104,412
 13번째 청크 완료 | 누적 원본 행 수: 13,000,000 | 현재 청크 OD 수: 103,757
 14번째 청크 완료 | 누적 원본 행 수: 14,000,000 | 현재 청크 OD 수: 104,346
 15번째 청크 완료 | 누적 원본 행 수: 15,000,000 | 현재 청크 OD 수: 103,520
 16번째 청크 완료 | 누적 원본 행 수: 16,000,000 | 현재 청크 OD 수: 103,599
 17번째 청크 완료 | 누적 원본 행 수: 17,000,000 | 현재 청크 OD 수: 103,618
 18번째 청크 완료 | 누적 원본 행 수

In [5]:
# =========================================================
# 4. 전체 청크 결과 통합
# =========================================================

od_partial = pd.concat(
    chunk_results,
    ignore_index=True,
)

print("청크 통합 전 중간 OD 행 수:", f"{len(od_partial):,}")


commute_od = (
    od_partial
    .groupby(
        GROUP_COLS,
        as_index=False,
        dropna=False,
        observed=True,
    )
    .agg(
        출근_이동량=("출근_이동량", "sum"),
        이동시간_가중합=("이동시간_가중합", "sum"),
        이동거리_가중합=("이동거리_가중합", "sum"),
        원본_행수=("원본_행수", "sum"),
    )
)


print("최종 OD 조합 수:", f"{len(commute_od):,}")

청크 통합 전 중간 OD 행 수: 10,360,151
최종 OD 조합 수: 164,860


In [6]:
# =========================================================
# 5. OD별 이동량 가중평균 계산
# =========================================================

commute_od["평균_이동시간_분"] = (
    commute_od["이동시간_가중합"]
    / commute_od["출근_이동량"]
)

commute_od["평균_이동거리_m"] = (
    commute_od["이동거리_가중합"]
    / commute_od["출근_이동량"]
)

commute_od["평균_이동거리_km"] = (
    commute_od["평균_이동거리_m"]
    / 1000
)


# 보기 좋게 반올림
commute_od["출근_이동량"] = (
    commute_od["출근_이동량"]
    .round(2)
)

commute_od["평균_이동시간_분"] = (
    commute_od["평균_이동시간_분"]
    .round(2)
)

commute_od["평균_이동거리_m"] = (
    commute_od["평균_이동거리_m"]
    .round(2)
)

commute_od["평균_이동거리_km"] = (
    commute_od["평균_이동거리_km"]
    .round(3)
)

In [7]:
# =========================================================
# 6. 결과 컬럼 정리
# =========================================================

commute_od = commute_od.rename(
    columns={
        "출발 행정동 코드": "거주동 코드",
        "출발 행정동": "거주동 이름",
        "도착 행정동 코드": "근무동 코드",
        "도착 행정동": "근무동 이름",
    }
)


FINAL_COLS = [
    "거주동 코드",
    "거주동 이름",
    "근무동 코드",
    "근무동 이름",
    "출근_이동량",
    "평균_이동시간_분",
    "평균_이동거리_m",
    "평균_이동거리_km",
    "원본_행수",
]


commute_od = commute_od[FINAL_COLS]


# 출근 이동량이 큰 순서로 정렬
commute_od = (
    commute_od
    .sort_values(
        "출근_이동량",
        ascending=False,
    )
    .reset_index(drop=True)
)


print("최종 결과 행 수:", f"{len(commute_od):,}")
display(commute_od.head(20))

최종 결과 행 수: 164,860


,거주동 코드,거주동 이름,근무동 코드,근무동 이름,출근_이동량,평균_이동시간_분,평균_이동거리_m,평균_이동거리_km,원본_행수
0,11560540,여의동,11560540,여의동,972322.53,14.15,622.50,0.622,14956
1,11545510,가산동,11545510,가산동,747881.32,15.10,770.26,0.770,13075
2,11680640,역삼1동,11680640,역삼1동,677469.04,13.72,550.29,0.550,13645
3,11545610,독산1동,11545510,가산동,630743.77,20.73,1378.76,1.379,12246
4,11560535,영등포동,11560540,여의동,593958.64,21.09,1535.68,1.536,12009
5,11650530,서초3동,11650530,서초3동,420971.92,14.60,561.68,0.562,12264
6,11380690,진관동,11380690,진관동,400772.79,19.01,817.23,0.817,12851
7,11560560,당산2동,11560540,여의동,368340.68,23.61,2021.70,2.022,10894
8,11530540,구로3동,11530540,구로3동,363509.18,12.46,417.65,0.418,10154
9,11500603,가양1동,11500603,가양1동,330015.10,13.88,834.38,0.834,9961


In [8]:
# =========================================================
# 7. 결과 기본 검증
# =========================================================

print("=" * 60)
print("OD 집계 결과 검증")
print("=" * 60)

print("OD 조합 수:", f"{len(commute_od):,}")

print(
    "총 출근 이동량:",
    f"{commute_od['출근_이동량'].sum():,.2f}"
)

print(
    "거주동 수:",
    f"{commute_od['거주동 코드'].nunique():,}"
)

print(
    "근무동 수:",
    f"{commute_od['근무동 코드'].nunique():,}"
)


# 중복 OD 확인
duplicate_od = commute_od.duplicated(
    subset=[
        "거주동 코드",
        "근무동 코드",
    ]
).sum()

print("중복 OD 수:", f"{duplicate_od:,}")


# 잘못된 대표값 확인
invalid_values = commute_od.loc[
    (commute_od["출근_이동량"] <= 0)
    | (commute_od["평균_이동시간_분"] < 0)
    | (commute_od["평균_이동거리_m"] < 0)
]

print(
    "0 이하 이동량 또는 음수 시간·거리:",
    f"{len(invalid_values):,}"
)


assert duplicate_od == 0, (
    "같은 거주동-근무동 조합이 중복되어 있습니다."
)

assert len(invalid_values) == 0, (
    "집계 결과에 잘못된 이동량·시간·거리 값이 있습니다."
)

print("기본 검증 완료")

OD 집계 결과 검증
OD 조합 수: 164,860
총 출근 이동량: 471,221,308.33
거주동 수: 428
근무동 수: 428
중복 OD 수: 0
0 이하 이동량 또는 음수 시간·거리: 0
기본 검증 완료


In [9]:
# =========================================================
# 8. 동일 행정동 내부 통근 확인
# =========================================================

same_dong_mask = (
    commute_od["거주동 코드"]
    == commute_od["근무동 코드"]
)

same_dong_od = commute_od.loc[same_dong_mask]


total_volume = commute_od["출근_이동량"].sum()
same_dong_volume = same_dong_od["출근_이동량"].sum()


print("동일 행정동 내부 OD 수:", f"{len(same_dong_od):,}")
print("동일 행정동 내부 출근량:", f"{same_dong_volume:,.2f}")

print(
    "전체 출근량 중 내부 통근 비율:",
    f"{same_dong_volume / total_volume * 100:.2f}%"
)

display(
    same_dong_od
    .sort_values(
        "출근_이동량",
        ascending=False,
    )
    .head(20)
)

동일 행정동 내부 OD 수: 428
동일 행정동 내부 출근량: 36,245,244.38
전체 출근량 중 내부 통근 비율: 7.69%


,거주동 코드,거주동 이름,근무동 코드,근무동 이름,출근_이동량,평균_이동시간_분,평균_이동거리_m,평균_이동거리_km,원본_행수
0,11560540,여의동,11560540,여의동,972322.53,14.15,622.50,0.622,14956
1,11545510,가산동,11545510,가산동,747881.32,15.10,770.26,0.770,13075
2,11680640,역삼1동,11680640,역삼1동,677469.04,13.72,550.29,0.550,13645
5,11650530,서초3동,11650530,서초3동,420971.92,14.60,561.68,0.562,12264
6,11380690,진관동,11380690,진관동,400772.79,19.01,817.23,0.817,12851
8,11530540,구로3동,11530540,구로3동,363509.18,12.46,417.65,0.418,10154
9,11500603,가양1동,11500603,가양1동,330015.10,13.88,834.38,0.834,9961
10,11110615,종로1.2.3.4가동,11110615,종로1.2.3.4가동,312526.36,14.86,414.10,0.414,11577
11,11440740,상암동,11440740,상암동,307522.13,16.45,718.67,0.719,11396
12,11650651,양재1동,11650651,양재1동,306250.55,14.37,792.47,0.792,10648


In [10]:
# =========================================================
# 9. 최종 결과 저장
# =========================================================

commute_od.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",
)

print("저장 완료:", OUTPUT_FILE)
print("파일 존재:", OUTPUT_FILE.exists())
print("저장 행 수:", f"{len(commute_od):,}")

print(
    "파일 크기:",
    f"{OUTPUT_FILE.stat().st_size / 1024**2:,.2f} MB"
)

저장 완료: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/all_age_commute_od_aggregated.csv
파일 존재: True
저장 행 수: 164,860
파일 크기: 11.20 MB
